[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch01/NN_DL_ch01_XORViz/NN_DL_ch01_XORViz.ipynb)

# NN_DL_ch01_XORViz

**Name:**  NN_DL_ch01_XORViz
**Published in:**  NN and Deep Learning with Business Applications
**Author:**  Daniel Traian Pele, ASE Bucharest / IDA
**Submitted:**  2026-03-24
**Description:**  Train MLP to solve XOR, visualize decision boundary and hidden representations
**Keywords:**  XOR, perceptron, decision boundary, MLP, hidden representation
**See also:**  NN_DL_ch01_CreditMLP


In [ ]:
# Install dependencies (Colab already has most)
# !pip install torch torchvision matplotlib numpy scikit-learn


In [ ]:

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ── Reproducibility ──────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Style constants ──────────────────────────────────────────────────────────
MAIN_BLUE = "#1A3A6E"
IDA_RED = "#CD0000"
FOREST = "#2E7D32"
CRIMSON = "#DC3545"

# ── 1. XOR data ─────────────────────────────────────────────────────────────
X = torch.tensor([[0.0, 0.0],
                   [0.0, 1.0],
                   [1.0, 0.0],
                   [1.0, 1.0]], dtype=torch.float32)
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

# ── 2. Model ────────────────────────────────────────────────────────────────
class XOR_MLP(nn.Module):
    """Minimal MLP to solve XOR: 2 → 4 (hidden) → 1."""

    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)
        self.output = nn.Linear(4, 1)

    def forward(self, x):
        h = torch.relu(self.hidden(x))
        return self.output(h)

    def hidden_repr(self, x):
        """Return the hidden-layer activations."""
        with torch.no_grad():
            return torch.relu(self.hidden(x)).numpy()


model = XOR_MLP()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

# ── 3. Training ─────────────────────────────────────────────────────────────
losses = []
for epoch in range(1, 2001):
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

model.eval()
with torch.no_grad():
    preds = torch.sigmoid(model(X))
print("XOR predictions:")
for xi, pi, ti in zip(X, preds, y):
    print(f"  {xi.tolist()} -> {pi.item():.4f}  (target {ti.item():.0f})")

# ── 4. Decision boundary mesh ───────────────────────────────────────────────
grid_res = 300
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, grid_res),
                      np.linspace(-0.5, 1.5, grid_res))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

with torch.no_grad():
    zz = torch.sigmoid(model(grid)).numpy().reshape(xx.shape)

# ── 5. Figure 1 — Loss curve ────────────────────────────────────────────────
fig0, ax0 = plt.subplots(figsize=(6, 3.5))
fig0.patch.set_alpha(0)
ax0.set_facecolor("none")
ax0.plot(losses, color=MAIN_BLUE, linewidth=1.3)
ax0.set_xlabel("Epoch", fontsize=11)
ax0.set_ylabel("BCE Loss", fontsize=11)
ax0.set_title("XOR MLP — Training Loss", fontsize=13, fontweight="bold", color=MAIN_BLUE)
ax0.spines["top"].set_visible(False)
ax0.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("nn_ch01_xor_loss.pdf", bbox_inches="tight", dpi=300, transparent=True)
plt.show()
print("Saved: nn_ch01_xor_loss.pdf")

# ── 6. Figure 2 — Decision boundary ─────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(6, 5.5))
fig1.patch.set_alpha(0)
ax1.set_facecolor("none")

contour = ax1.contourf(xx, yy, zz, levels=50, cmap="RdBu_r", alpha=0.7)
ax1.contour(xx, yy, zz, levels=[0.5], colors=["black"], linewidths=2)

# Data points
for i in range(4):
    marker = "o" if y[i].item() == 1 else "s"
    color = FOREST if y[i].item() == 1 else CRIMSON
    ax1.scatter(X[i, 0], X[i, 1], c=color, s=200, marker=marker,
                edgecolors="black", linewidths=1.5, zorder=5)

ax1.set_xlabel("$x_1$", fontsize=12)
ax1.set_ylabel("$x_2$", fontsize=12)
ax1.set_title("XOR — Decision Boundary", fontsize=13, fontweight="bold", color=MAIN_BLUE)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=FOREST,
           markersize=10, markeredgecolor="black", label="Class 1 (XOR=1)"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor=CRIMSON,
           markersize=10, markeredgecolor="black", label="Class 0 (XOR=0)"),
]
ax1.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.10),
           ncol=2, frameon=False, fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("nn_ch01_xor_boundary.pdf", bbox_inches="tight", dpi=300, transparent=True)
plt.show()
print("Saved: nn_ch01_xor_boundary.pdf")

# ── 7. Figure 3 — Hidden-layer representations ──────────────────────────────
H = model.hidden_repr(X)  # (4, 4) — project first two hidden units

fig2, ax2 = plt.subplots(figsize=(6, 5))
fig2.patch.set_alpha(0)
ax2.set_facecolor("none")

for i in range(4):
    color = FOREST if y[i].item() == 1 else CRIMSON
    marker = "o" if y[i].item() == 1 else "s"
    label_text = f"({X[i,0]:.0f},{X[i,1]:.0f})→{y[i].item():.0f}"
    ax2.scatter(H[i, 0], H[i, 1], c=color, s=200, marker=marker,
                edgecolors="black", linewidths=1.5, zorder=5)
    ax2.annotate(label_text, (H[i, 0], H[i, 1]),
                 textcoords="offset points", xytext=(8, 8), fontsize=9,
                 fontweight="bold", color=color)

ax2.set_xlabel("Hidden unit 1", fontsize=11)
ax2.set_ylabel("Hidden unit 2", fontsize=11)
ax2.set_title("XOR — Hidden-Layer Representation", fontsize=13,
              fontweight="bold", color=MAIN_BLUE)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("nn_ch01_xor_hidden.pdf", bbox_inches="tight", dpi=300, transparent=True)
plt.show()
print("Saved: nn_ch01_xor_hidden.pdf")

